# Supplier Risk Assessment & Supply Chain Disruption Cost Analysis

## To identify high risk suppliers based on delivery relibility, quality and lead time varibility, assess supplier dependency through concentration risk, and quantify financial impact of potential disruptions to recommend strategies that minimize supply chain risk and cost 

## Phase 1 Load Data

In [ ]:
import pandas as pd

df = pd.read_csv("Procurement KPI Analysis Dataset.csv")

df.head()

## Structure Inspection

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

# Phase 2 Feature Engineering

## Convert Dates 

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'])

## Create Lead Time

In [ ]:
df['lead_time'] = (df['Delivery_Date'] - df['Order_Date']).dt.days

## Handle Order Status

In [ ]:
df = df[df['Order_Status'] == 'Delivered'].copy()

## On_Time Delivery

#### Delivery <= 10days = On-time

In [ ]:
df['on_time_flag'] = df['lead_time'] <= 10

In [ ]:
on_time = df.groupby('Supplier')['on_time_flag'].mean().reset_index()
on_time.columns = ['Supplier', 'on_time_pct']

## //Handle NaN in Defective_Units//

In [ ]:
df['Defective_Units'] = df['Defective_Units'].fillna(0)

## //Check divide-by-zero//

In [ ]:
df = df[df['Quantity'] > 0]

## Defect Rate

In [ ]:
df['defect_rate'] = df['Defective_Units']/ df['Quantity']

In [ ]:
defect = df.groupby('Supplier')['defect_rate'].mean().reset_index()

## Lead Time Varibility

In [ ]:
lt_var = df.groupby('Supplier')['lead_time'].std().reset_index()
lt_var.columns = ['Supplier', 'lt_variability']

## Spend Calculation

In [ ]:
df['spend'] = df['Quantity'] * df['Negotiated_Price']

In [ ]:
spend = df.groupby('Supplier')['spend'].sum().reset_index()

## Merge All Metrics

In [ ]:
final_df = on_time.merge(defect, on = 'Supplier') \
                  .merge(lt_var, on = 'Supplier') \
                  .merge(spend, on = 'Supplier')

In [ ]:
final_df.head()

## Normalization 

In [ ]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

final_df['on_time_norm'] = normalize(final_df['on_time_pct'])
final_df['defect_norm'] = normalize(final_df['defect_rate'])
final_df['lt_var_norm'] = normalize(final_df['lt_variability'])

In [ ]:
final_df.head()

In [ ]:
final_df.rename(columns={'lt_variablility': 'lead_time_variability'}, inplace=True)

## Concentration Risk(simulated)

## Category Level Spend

In [ ]:
df['spend'] = df['Quantity'] * df['Negotiated_Price']

category_spend = df.groupby(['Item_Category', 'Supplier'])['spend'].sum().reset_index()

## Total Spend per Category

In [ ]:
category_spend['total_category_spend'] = category_spend.groupby('Item_Category')['spend'].transform('sum')

## Supplier Share 

In [ ]:
category_spend['supplier_share'] = category_spend['spend'] / category_spend['total_category_spend']

category_spend.head()

## Assign Risk Categories + Ranking

In [ ]:
# Define risk levels

def classify(x):
    if x == 1:
        return 'Critical'
    elif x > 0.7:
        return 'High'
    elif x > 0.5:
        return 'Medium'
    else:
        return 'Low'

category_spend['concentration_risk'] = category_spend['supplier_share'].apply(classify)

# Map to numeric ranking 
risk_order = {
    'Low' : 1,
    'Medium' : 2,
    'High' : 3,
    'Critical' : 4
}

category_spend['risk_value'] = category_spend['concentration_risk'].map(risk_order)

category_spend.head()

## Maximum Risk Per Supplier

In [ ]:
# Highest Risk Level per supplier

supplier_conc = category_spend.groupby('Supplier')['risk_value'].max().reset_index()

supplier_conc.head()

## Convert Back to Category Labels

In [ ]:
# Revese Mapping

inverse_risk_order = {v: k for k, v in risk_order.items()}

supplier_conc['concentration_risk'] = supplier_conc['risk_value'].map(inverse_risk_order)

supplier_conc = supplier_conc.drop(columns='risk_value')

supplier_conc.head()

In [ ]:
## Merge With Main Dataset

final_df = final_df.merge(supplier_conc, on='Supplier')

final_df.head()

In [ ]:
# Check Distribution of concentration of Risk
final_df['concentration_risk'].value_counts(normalize=True) * 100

## Phase 4: Risk Scoring

## Apply correct risk formula

In [ ]:
final_df['risk_score'] = (
    0.4 * (1 - final_df['on_time_norm']) +
    0.3 * final_df['defect_norm'] +
    0.3 * final_df['lt_var_norm']
)

## Categorize Risk

In [ ]:
def categorize(r):
    if r < 0.33:
        return 'Green'
    elif r < 0.66:
        return 'Yellow'
    else:
        return 'Red'

final_df['risk_category'] = final_df['risk_score'].apply(categorize)

## Rank Supplier

In [ ]:
final_df = final_df.sort_values(by='risk_score', ascending = False)

## View Result

In [ ]:
final_df[['Supplier', 'risk_score', 'risk_category']].head()

## Quick Insight Check

In [ ]:
final_df['risk_category'].value_counts()

## Phase 5: Expected Loss Model

In [ ]:
# Business Assumptions

daily_production_loss = 50000   # rs per day
avg_delay_days = 5            # average disruption duration

daily_production_loss, avg_delay_days

In [ ]:
# Current Expected Loss
final_df['expected_loss'] = (
    final_df['risk_score'] *
    daily_production_loss *
    avg_delay_days
)

final_df[['Supplier', 'risk_score', 'expected_loss']]

In [ ]:
# Total Current Risk Exposure

total_current_risk = final_df['expected_loss'].sum()

print(f"Total Annual Risk Exposure: Rs {total_current_risk:,.0f}")

## Apply Risk Reduction Strategy

In [ ]:
# Reduce Risk By 40% for High Risk Suppliers (risk score > 0.7)

final_df['risk_score_proposed'] = final_df['risk_score'].apply(
    lambda x: x * 0.6 if x > 0.7 else x
)

final_df[['Supplier', 'risk_score', 'risk_score_proposed']]

## Proposed Expected Loss 

In [ ]:
# Expected loss after improvements
final_df['expected_loss_proposed'] = (
    final_df['risk_score_proposed'] *
    daily_production_loss *
    avg_delay_days
)

final_df[['Supplier', 'expected_loss', 'expected_loss_proposed']]

## Total Proposed Risk

In [ ]:
# Total risk after improvement

total_proposed_risk = final_df['expected_loss_proposed'].sum()

print(f"Total Risk After Improvement: Rs{total_proposed_risk:,.0f}")

## Calculated Saving

In [ ]:
# Risk reduction (saving)

risk_reduction = total_current_risk - total_proposed_risk

print(f"Total Savings: Rs {risk_reduction:,.0f}")

## Payback Period Calculation

In [ ]:
# Assume implementation cost
implementation_cost = 200000    #Rs

payback_period = implementation_cost / risk_reduction

print(f"Payback Period: {payback_period:.2f} years")

## Final Summary Table 

In [ ]:
final_df[['Supplier', 'risk_score', 'expected_loss', 'risk_score_proposed', 'expected_loss_proposed']]

## Export Final Data

In [ ]:
final_df.to_excel("supplier_risk_final.xlsx", index = False)